MARKDOWN PLACEHOLDER

In [9]:
# Import all related stuff and make Supabase connection
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd

load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

In [ ]:
# Sales table from Supabase
sales_response = supabase.table("sales").select("*").execute()
df_sales = pd.DataFrame(sales_response.data)
df_sales.head()
# df_sales.shape
# df_sales.dtypes

id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object

In [ ]:
# Products table from Supabase
products_response = supabase.table("products").select("*").execute()
df_products = pd.DataFrame(products_response.data)
df_products.head()
# df_products.shape
# df_products.dtypes

product_id         int64
product_name         str
category             str
subcategory          str
supplier             str
cost_price       float64
retail_price     float64
eco_certified     object
created_at           str
dtype: object

In [17]:
# Customers table from Supabase
customers_response = supabase.table("customers").select("*").execute()
df_customers = pd.DataFrame(customers_response.data)
df_customers.head()
# df_customers.shape
# df_customers.dtypes

,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [ ]:
# Merge Sales, Products and Customers tables into one df
df = (
    df_sales
    .merge(df_products, on='product_id', how='left')
    .merge(df_customers, on='customer_id', how='left')
)
df.head()
# df.shape
# df.dtypes

id                     int64
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
product_name             str
category                 str
subcategory              str
supplier                 str
cost_price           float64
retail_price         float64
eco_certified         object
created_at               str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object

In [ ]:
# 0. OLULINE: Teisendame algse tabeli sale_date veeru kuupäevaks!
df['sale_date'] = pd.to_datetime(df['sale_date'])

# 1. Viitekuupäev
today = pd.to_datetime('2025-02-28')

# 2. Recency — viimase ostu kuupäev ja päevade arv tänaseni
recency_df = df.groupby('customer_id')['sale_date'].max().reset_index()
recency_df['recency_days'] = (today - recency_df['sale_date']).dt.days

# 3. Frequency — ostude arv
frequency_df = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency_df.columns = ['customer_id', 'frequency']

# 4. Monetary — kogukulutus
monetary_df = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary_df.columns = ['customer_id', 'monetary_value']

# 5. Liidame R, F, M ühte rfm tabelisse (pd.merge)
rfm = pd.merge(
    recency_df[['customer_id', 'recency_days']], frequency_df, on='customer_id'
)
rfm = pd.merge(rfm, monetary_df, on='customer_id')

# 6. Määrame skoorid 1-5 (pd.qcut)
# Recency: vastupidine (vähem päevi = skoor 5)
rfm['R_score'] = pd.qcut(
    rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: vajab rank(method='first') duplikaatide tõttu
rfm['F_score'] = (
    pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
    .astype(int)
)

# Monetary: suurem summa = skoor 5
rfm['M_score'] = pd.qcut(
    rfm['monetary_value'], 5, labels=[1, 2, 3, 4, 5]
).astype(int)

# 7. Arvutame koguskoori ja määrame segmendid
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']


def get_segment(score):
  if score >= 13:
    return 'VIP Champions'
  elif score >= 10:
    return 'Loyal'
  elif score >= 7:
    return 'Potential'
  elif score >= 4:
    return 'At Risk'
  else:
    return 'Lost'


rfm['Segment'] = rfm['RFM_Score'].apply(get_segment)

# 8. Trükime segmentide kokkuvõtte
summary = pd.DataFrame({
    'Arv': rfm['Segment'].value_counts(),
    'Osakaal (%)': (rfm['Segment'].value_counts(normalize=True) * 100).round(1),
})

print('=== RFM SEGMENTIDE KOKKUVÕTE ===')
print(summary)

=== RFM SEGMENTIDE KOKKUVÕTE ===
               Arv  Osakaal (%)
Segment                        
Potential      770         30.2
Loyal          691         27.1
At Risk        535         21.0
VIP Champions  446         17.5
Lost           109          4.3
